# Calendar Dimension Table

Builds the `adb_caresync.silver.calender` date dimension with one row per day from 1940-01-01 to 2030-12-31, including standard date attributes (date key, year, quarter, month, week, day, weekend flags, period boundaries).

In [0]:
%sql
-- Calendar dimension: one row per day from 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE adb_caresync.silver.calender
COMMENT 'Calendar date dimension, 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  CAST(date_format(date, 'yyyyMMdd') AS INT)              AS date_key,
  date                                                    AS date,
  -- Year
  year(date)                                              AS year,
  CASE WHEN (year(date) % 4 = 0 AND year(date) % 100 <> 0) OR year(date) % 400 = 0
       THEN true ELSE false END                           AS is_leap_year,
  DATE_TRUNC('YEAR', date)::DATE                          AS first_day_of_year,
  make_date(year(date), 12, 31)                           AS last_day_of_year,
  -- Quarter
  quarter(date)                                           AS quarter,
  concat('Q', quarter(date))                              AS quarter_name,
  concat(year(date), '-Q', quarter(date))                 AS year_quarter,
  DATE_TRUNC('QUARTER', date)::DATE                       AS first_day_of_quarter,
  date_sub(add_months(DATE_TRUNC('QUARTER', date)::DATE, 3), 1) AS last_day_of_quarter,
  -- Month
  month(date)                                             AS month,
  date_format(date, 'MMMM')                               AS month_name,
  date_format(date, 'MMM')                                AS month_short_name,
  date_format(date, 'yyyy-MM')                            AS year_month,
  CAST(date_format(date, 'yyyyMM') AS INT)                AS year_month_key,
  DATE_TRUNC('MONTH', date)::DATE                         AS first_day_of_month,
  last_day(date)                                          AS last_day_of_month,
  day(last_day(date))                                     AS days_in_month,
  -- Week (ISO 8601: weeks start Monday)
  weekofyear(date)                                        AS week_of_year,
  CAST(ceil(day(date) / 7.0) AS INT)                      AS week_of_month,
  DATE_TRUNC('WEEK', date)::DATE                          AS week_start_date,
  date_add(DATE_TRUNC('WEEK', date)::DATE, 6)             AS week_end_date,
  -- Day
  day(date)                                               AS day_of_month,
  dayofyear(date)                                         AS day_of_year,
  weekday(date) + 1                                       AS day_of_week,       -- 1 = Monday ... 7 = Sunday
  date_format(date, 'EEEE')                               AS day_name,
  date_format(date, 'EEE')                                AS day_short_name,
  CASE WHEN weekday(date) >= 5 THEN true ELSE false END   AS is_weekend,
  CASE WHEN weekday(date) < 5  THEN true ELSE false END   AS is_weekday,
  CASE WHEN date = last_day(date) THEN true ELSE false END AS is_month_end,
  -- Half year / decade
  CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END            AS half_year,
  CAST(floor(year(date) / 10) * 10 AS INT)                AS decade
FROM dates
ORDER BY date;

In [0]:
%sql
-- Validation: row count, range, uniqueness
SELECT
  count(*)                         AS total_rows,
  count(DISTINCT date_key)         AS distinct_keys,
  min(date)                        AS min_date,
  max(date)                        AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_rows
FROM adb_caresync.silver.calender;

In [0]:
%sql
SELECT * FROM adb_caresync.silver.calender WHERE date BETWEEN DATE'2024-12-28' AND DATE'2025-01-05' ORDER BY date;